In [0]:
from pyspark.sql.functions import current_timestamp

df_orders = spark.read.csv(
    "/Volumes/teste_koin/koin_origem/raw_data/BRA - Cientista de dados - Orders.csv",
    header=True,
    inferSchema=True,
    sep=",",
    encoding="UTF-8"
)

df_orders.withColumn("ingestion_date", current_timestamp()) \
  .write \
  .format("delta") \
  .mode("append") \
  .option("overwriteSchema", "true") \
  .saveAsTable("teste_koin.default.bronze_orders")

In [0]:
from pyspark.sql.functions import current_timestamp

CATALOG_SCHEMA = "teste_koin.default"
RAW_VOLUME = "/Volumes/teste_koin/default/raw"

def ingest_to_bronze(file_name, table_name):
    df = spark.read.csv(
        f"{RAW_VOLUME}/{file_name}",
        header=True,
        inferSchema=True,
        sep=",",
        encoding="UTF-8"
    )

    df_bronze = df.withColumn("ingestion_timestamp", current_timestamp())

    full_table_name = f"{CATALOG_SCHEMA}.bronze_{table_name}"

    df_bronze.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable(full_table_name)

    print(f"Dados {file_name} adicionados com sucesso!")


ingest_to_bronze(
    file_name="BRA - Cientista de dados - Orders.csv",
    table_name="customers"
)